# Backfill BTC / ETH 5-minute TWAP60 strikes — no subscription

This notebook retrieves each market's official **Price to Beat** from
Polymarket's public Gamma event metadata. It requires no Chainlink account,
API key, wallet, or Polymarket authentication.

It preserves the existing strike CSV filenames and the exact 17-column
schema/order used by `--reference-strikes-csv`:

- `btc_5m_strikes.csv`
- `eth_5m_strikes.csv`

Safety rules:

- Only explicit Gamma `priceToBeat` / `price_to_beat` fields are accepted.
- The parser covers event, `eventMetadata`, event `metadata`, market, and
  market `metadata` locations, including the field missed by the old parser.
- Multiple populated fields must agree numerically; conflicts fail closed.
- Missing, malformed, wrong-slug, or ambiguous responses leave the strike
  blank. There is no candle-open, spot-price, local-TWAP, or settlement-based
  fallback.
- Existing `polymarket_crypto_price_api:*` rows are actively replaced in the
  selected date range, even when their wrong strike is already populated.
- Existing `gamma_field:priceToBeat` rows and rows outside the selected range
  are preserved by default.
- A timestamped backup is created before an existing CSV is replaced.

This is for markets whose rules specify **Chainlink BTC/USD or ETH/USD
TWAP60**. It must not be used for older contracts with different reference
definitions.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import shutil
import time
from datetime import datetime, timedelta, timezone
from decimal import Decimal, InvalidOperation
from email.utils import parsedate_to_datetime
from pathlib import Path
from typing import Any

import pandas as pd
import requests


# ============================================================
# CONFIG
# ============================================================

# STRIKES_DIR lets GitHub Actions point this at the repo's data folder;
# locally it falls back to your Windows path.
DATA_PATH = Path(os.environ.get(
    "STRIKES_DIR",
    r"C:\Wealth Project\db\polymarket\polymarket_reference",
))

# Choose exactly one: "btc", "eth", or "both".
ASSET_MODE = os.environ.get("ASSET_MODE", "both")

# Inclusive UTC dates. None means the last completed bucket before now - 1h.
START_DATE = "2026-08-30"
END_DATE = None

STRIKE_FILE_NAMES = {
    "btc": "btc_5m_strikes.csv",
    "eth": "eth_5m_strikes.csv",
}

GAMMA_EVENT_BY_SLUG_URL = (
    "https://gamma-api.polymarket.com/events/slug"
)
BUCKET_SECONDS = 300

REQUEST_TIMEOUT_SECONDS = 20
MAX_ATTEMPTS = 4
SLEEP_SECONDS = 0.10
CHECKPOINT_EVERY = 25
PROGRESS_UPDATE_SECONDS = 1.0
# Git already keeps history in CI, so backups are off there.
CREATE_BACKUP = os.environ.get("CREATE_BACKUP", "1") == "1"
FETCH_CLOSE_PRICE = True

# Repair old wrong-source rows as well as rows with missing strikes.
REPAIR_SOURCE_PREFIXES = (
    "polymarket_crypto_price_api:",
)
REPLACE_ALL_ROWS_IN_RANGE = False

# Use an integer (for example 2) for a small first run.
# None processes every repair target in the configured range.
MAX_FETCH_ROWS: int | None = None

RUN_OFFLINE_SELF_TESTS = True
RUN_KNOWN_BUCKET_VALIDATION = True
KNOWN_BUCKET_VALIDATION = {
    # Official Polymarket page displays 78,421.77. The live exact E18
    # observation was 78421.765923971878879232.
    ("btc", 1789402200): Decimal("78421.77"),
}
KNOWN_BUCKET_TOLERANCE_USD = Decimal("0.01")

# False keeps notebook output compact; the live progress line remains.
PRINT_EACH_SUCCESS = False
PRINT_FIRST_N_FAILURES = 30

GAMMA_VARIANT = "twap60_gamma_priceToBeat"
GAMMA_SOURCE = "gamma_field:priceToBeat"


def selected_underlyings(asset_mode: str) -> list[str]:
    mode = str(asset_mode).strip().lower()
    if mode == "btc":
        return ["btc"]
    if mode == "eth":
        return ["eth"]
    if mode == "both":
        return ["btc", "eth"]
    raise ValueError(
        'ASSET_MODE must be "btc", "eth", or "both". '
        f"Got: {asset_mode!r}"
    )


SELECTED_UNDERLYINGS = selected_underlyings(ASSET_MODE)

print(f"Asset mode:              {ASSET_MODE}")
print(f"Selected underlyings:    {SELECTED_UNDERLYINGS}")
print(f"Reference directory:     {DATA_PATH}")
print("Credentials required:    none")
for _underlying in SELECTED_UNDERLYINGS:
    print(
        f"{_underlying.upper()} strike file:         "
        f"{DATA_PATH / STRIKE_FILE_NAMES[_underlying]}"
    )


In [ ]:
# ============================================================
# TIME, SCHEMA, AND FILE HELPERS
# ============================================================

def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def floor_to_5m(unix_ts: int) -> int:
    return unix_ts - (unix_ts % BUCKET_SECONDS)


def parse_date_to_utc_start(date_str: str) -> datetime:
    return datetime.strptime(
        date_str,
        "%Y-%m-%d",
    ).replace(tzinfo=timezone.utc)


def resolve_end_datetime(end_date: str | None) -> datetime:
    if end_date is None:
        return utc_now() - timedelta(hours=1)
    return (
        parse_date_to_utc_start(end_date)
        + timedelta(days=1)
        - timedelta(seconds=1)
    )


def bucket_range_from_dates(
    start_date: str,
    end_date: str | None,
) -> list[int]:
    start_dt = parse_date_to_utc_start(start_date)
    end_dt = resolve_end_datetime(end_date)
    start_ts = floor_to_5m(int(start_dt.timestamp()))
    end_ts = floor_to_5m(int(end_dt.timestamp()))
    if end_ts < start_ts:
        raise ValueError(
            "Resolved end timestamp is earlier than start timestamp."
        )
    return list(range(start_ts, end_ts + 1, BUCKET_SECONDS))


def unix_to_iso_utc(unix_ts: int) -> str:
    return (
        datetime.fromtimestamp(unix_ts, tz=timezone.utc)
        .replace(microsecond=0)
        .isoformat()
        .replace("+00:00", "Z")
    )


def make_slug(underlying: str, bucket_start_ts: int) -> str:
    return f"{underlying.lower()}-updown-5m-{bucket_start_ts}"


BASE_COLUMNS = [
    "bucket_start_ts",
    "underlying",
    "strike",
    "slug",
    "source",
    "fetched_at_utc",
]

EXTRA_COLUMNS = [
    "close_price",
    "crypto_price_variant",
    "crypto_price_request_url",
    "crypto_price_http_status",
    "crypto_price_timestamp_ms",
    "crypto_price_completed",
    "crypto_price_incomplete",
    "crypto_price_cached",
    "crypto_price_response_json",
    "crypto_price_response_text",
    "crypto_price_error",
]

ALL_COLUMNS = BASE_COLUMNS + EXTRA_COLUMNS


def empty_cache() -> pd.DataFrame:
    return pd.DataFrame(columns=ALL_COLUMNS)


def normalize_cache(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ALL_COLUMNS:
        if col not in out.columns:
            out[col] = pd.NA

    out["bucket_start_ts"] = pd.to_numeric(
        out["bucket_start_ts"], errors="coerce"
    ).astype("Int64")
    out["underlying"] = (
        out["underlying"].astype("string").str.lower()
    )
    out["strike"] = pd.to_numeric(
        out["strike"], errors="coerce"
    )
    out["close_price"] = pd.to_numeric(
        out["close_price"], errors="coerce"
    )
    out["crypto_price_http_status"] = pd.to_numeric(
        out["crypto_price_http_status"], errors="coerce"
    ).astype("Int64")
    out["crypto_price_timestamp_ms"] = pd.to_numeric(
        out["crypto_price_timestamp_ms"], errors="coerce"
    ).astype("Int64")

    for col in (
        "slug",
        "source",
        "fetched_at_utc",
        "crypto_price_variant",
        "crypto_price_request_url",
        "crypto_price_response_json",
        "crypto_price_response_text",
        "crypto_price_error",
    ):
        out[col] = out[col].astype("string")
    return out[ALL_COLUMNS]


def load_cache(path: Path) -> pd.DataFrame:
    if not path.exists():
        return empty_cache()
    return normalize_cache(pd.read_csv(path))


def source_needs_repair(source_value: Any) -> bool:
    text = "" if pd.isna(source_value) else str(source_value)
    return any(
        text.startswith(prefix)
        for prefix in REPAIR_SOURCE_PREFIXES
    )


def row_needs_repair(row: pd.Series) -> bool:
    if REPLACE_ALL_ROWS_IN_RANGE:
        return True
    return bool(
        pd.isna(row.get("strike"))
        or source_needs_repair(row.get("source"))
    )


def build_repair_targets(
    base_df: pd.DataFrame,
    underlying: str,
    bucket_timestamps: list[int],
) -> list[int]:
    normalized = normalize_cache(base_df)
    existing = {
        int(row["bucket_start_ts"]): row
        for _, row in normalized.iterrows()
        if not pd.isna(row["bucket_start_ts"])
        and (
            pd.isna(row["underlying"])
            or str(row["underlying"]).lower() == underlying.lower()
        )
    }

    targets: list[int] = []
    for bucket_ts in bucket_timestamps:
        row = existing.get(int(bucket_ts))
        if row is None or row_needs_repair(row):
            targets.append(int(bucket_ts))
    if MAX_FETCH_ROWS is not None:
        targets = targets[: int(MAX_FETCH_ROWS)]
    return targets


def merge_fetched_rows(
    base_df: pd.DataFrame,
    fetched_df: pd.DataFrame,
) -> pd.DataFrame:
    base = normalize_cache(base_df)
    fetched = normalize_cache(fetched_df)
    combined = pd.concat(
        [base, fetched], ignore_index=True, sort=False
    )
    combined = combined.drop_duplicates(
        subset=["bucket_start_ts"], keep="last"
    )
    return (
        normalize_cache(combined)
        .sort_values("bucket_start_ts", kind="mergesort")
        .reset_index(drop=True)
    )


def make_timestamped_backup(path: Path) -> Path | None:
    if not CREATE_BACKUP or not path.exists():
        return None
    stamp = utc_now().strftime("%Y%m%dT%H%M%SZ")
    backup = path.with_name(
        f"{path.stem}.pre_gamma_twap60_{stamp}{path.suffix}"
    )
    counter = 1
    while backup.exists():
        backup = path.with_name(
            f"{path.stem}.pre_gamma_twap60_{stamp}_{counter}"
            f"{path.suffix}"
        )
        counter += 1
    shutil.copy2(path, backup)
    return backup


def save_output(df: pd.DataFrame, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    out = (
        normalize_cache(df)
        .sort_values("bucket_start_ts", kind="mergesort")
        .drop_duplicates(subset=["bucket_start_ts"], keep="last")
    )
    temp_path = output_path.with_suffix(output_path.suffix + ".tmp")
    out.to_csv(temp_path, index=False)
    temp_path.replace(output_path)


In [ ]:
# ============================================================
# PUBLIC POLYMARKET GAMMA CLIENT
# ============================================================

RETRYABLE_HTTP_STATUSES = {
    408, 425, 429, 500, 502, 503, 504,
}


class GammaRequestError(RuntimeError):
    def __init__(
        self,
        message: str,
        *,
        status_code: int | None = None,
        request_url: str | None = None,
        response_text: str | None = None,
        retryable: bool = False,
    ) -> None:
        super().__init__(message)
        self.status_code = status_code
        self.request_url = request_url
        self.response_text = response_text
        self.retryable = retryable


SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/150.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json",
    "Referer": "https://polymarket.com/",
})


def retry_after_seconds(
    response: requests.Response,
) -> float | None:
    value = response.headers.get("Retry-After")
    if not value:
        return None
    try:
        return max(0.0, float(value))
    except ValueError:
        pass
    try:
        retry_dt = parsedate_to_datetime(value)
        if retry_dt.tzinfo is None:
            retry_dt = retry_dt.replace(tzinfo=timezone.utc)
        return max(0.0, (retry_dt - utc_now()).total_seconds())
    except Exception:
        return None


def nested_get(payload: dict[str, Any], *path: str) -> Any:
    current: Any = payload
    for key in path:
        if not isinstance(current, dict):
            return None
        current = current.get(key)
    return current


def scalar_text(value: Any) -> str:
    if isinstance(value, str):
        return value
    if isinstance(value, Decimal):
        return str(value)
    return json.dumps(
        value,
        ensure_ascii=False,
        separators=(",", ":"),
    )


def positive_decimal(value: Any, field_path: str) -> Decimal:
    if isinstance(value, bool) or value is None:
        raise ValueError(f"{field_path} is not numeric")
    raw = str(value).strip()
    if not re.fullmatch(
        r"[+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?",
        raw,
    ):
        raise ValueError(
            f"{field_path} is not a plain numeric scalar: {value!r}"
        )
    try:
        parsed = Decimal(raw)
    except InvalidOperation as exc:
        raise ValueError(
            f"{field_path} is not numeric: {value!r}"
        ) from exc
    if not parsed.is_finite() or parsed <= 0:
        raise ValueError(
            f"{field_path} is not positive and finite: {value!r}"
        )
    return parsed


PRICE_TO_BEAT_PATHS = (
    ("market.priceToBeat", ("priceToBeat",)),
    ("market.price_to_beat", ("price_to_beat",)),
    ("market.metadata.priceToBeat", ("metadata", "priceToBeat")),
    ("market.metadata.price_to_beat", ("metadata", "price_to_beat")),
    ("event.priceToBeat", ("priceToBeat",)),
    ("event.price_to_beat", ("price_to_beat",)),
    (
        "event.eventMetadata.priceToBeat",
        ("eventMetadata", "priceToBeat"),
    ),
    (
        "event.eventMetadata.price_to_beat",
        ("eventMetadata", "price_to_beat"),
    ),
    ("event.metadata.priceToBeat", ("metadata", "priceToBeat")),
    (
        "event.metadata.price_to_beat",
        ("metadata", "price_to_beat"),
    ),
)


def collect_price_to_beat_candidates(
    event: dict[str, Any],
    market: dict[str, Any],
) -> list[dict[str, Any]]:
    candidates: list[dict[str, Any]] = []
    for field_path, path in PRICE_TO_BEAT_PATHS:
        root = market if field_path.startswith("market.") else event
        value = nested_get(root, *path)
        if value is None:
            continue
        candidate: dict[str, Any] = {
            "field": field_path,
            "raw": scalar_text(value),
            "value": None,
            "valid": False,
        }
        try:
            parsed = positive_decimal(value, field_path)
            candidate["value"] = parsed
            candidate["valid"] = True
        except ValueError as exc:
            candidate["error"] = str(exc)
        candidates.append(candidate)
    return candidates


def select_price_to_beat(
    event: dict[str, Any],
    market: dict[str, Any],
) -> tuple[Decimal, str, str, list[dict[str, Any]]]:
    candidates = collect_price_to_beat_candidates(event, market)
    invalid = [item for item in candidates if not item["valid"]]
    if invalid:
        details = "; ".join(
            f"{item['field']}={item['raw']!r}"
            for item in invalid
        )
        raise ValueError(
            f"Malformed populated priceToBeat field(s): {details}"
        )
    valid = [item for item in candidates if item["valid"]]
    if not valid:
        raise ValueError(
            "Gamma response has no explicit priceToBeat field"
        )
    distinct = {item["value"] for item in valid}
    if len(distinct) != 1:
        details = "; ".join(
            f"{item['field']}={item['raw']}"
            for item in valid
        )
        raise ValueError(
            f"Conflicting Gamma priceToBeat fields: {details}"
        )
    selected = valid[0]
    audit_candidates = [
        {
            "field": item["field"],
            "raw": item["raw"],
            "value": str(item["value"]),
        }
        for item in valid
    ]
    return (
        selected["value"],
        selected["field"],
        selected["raw"],
        audit_candidates,
    )


def normalize_event_payload(
    payload: Any,
    expected_slug: str,
) -> tuple[dict[str, Any], dict[str, Any]]:
    if isinstance(payload, list):
        if len(payload) != 1 or not isinstance(payload[0], dict):
            raise ValueError(
                "Gamma response must contain exactly one event"
            )
        event = payload[0]
    elif isinstance(payload, dict):
        event = payload
    else:
        raise ValueError("Gamma response is not an event object")

    actual_slug = str(event.get("slug", "")).strip()
    if actual_slug != expected_slug:
        raise ValueError(
            f"Gamma slug mismatch: expected {expected_slug!r}, "
            f"got {actual_slug!r}"
        )
    markets = event.get("markets")
    if not isinstance(markets, list) or len(markets) != 1:
        raise ValueError(
            "Expected exactly one market in the 5-minute event"
        )
    market = markets[0]
    if not isinstance(market, dict):
        raise ValueError("Gamma market is not an object")
    return event, market


def fetch_gamma_price_to_beat(
    underlying: str,
    bucket_start_ts: int,
    *,
    max_attempts: int = MAX_ATTEMPTS,
    timeout_seconds: float = REQUEST_TIMEOUT_SECONDS,
) -> dict[str, Any]:
    slug = make_slug(underlying, bucket_start_ts)
    request_url = f"{GAMMA_EVENT_BY_SLUG_URL}/{slug}"
    last_error: Exception | None = None

    for attempt in range(1, max_attempts + 1):
        response: requests.Response | None = None
        try:
            response = SESSION.get(
                request_url,
                timeout=timeout_seconds,
            )
            status = int(response.status_code)
            response_text = response.text
            if status != 200:
                retryable = status in RETRYABLE_HTTP_STATUSES
                raise GammaRequestError(
                    f"Gamma HTTP {status}",
                    status_code=status,
                    request_url=str(response.url),
                    response_text=response_text[:2000],
                    retryable=retryable,
                )

            # Preserve every decimal digit if Gamma emits the value as
            # a JSON number instead of a string.
            payload = json.loads(
                response_text,
                parse_float=Decimal,
            )
            event, market = normalize_event_payload(payload, slug)
            value, field, raw, candidates = select_price_to_beat(
                event, market
            )
            fetched_at = utc_now()
            response_sha256 = hashlib.sha256(
                response.content
            ).hexdigest()
            condition_id = (
                market.get("conditionId")
                or market.get("condition_id")
                or market.get("conditionID")
            )
            audit = {
                "slug": slug,
                "underlying": underlying.lower(),
                "bucket_start_ts": int(bucket_start_ts),
                "priceToBeat": raw,
                "priceToBeatField": field,
                "priceToBeatCandidates": candidates,
                "conditionId": (
                    str(condition_id)
                    if condition_id is not None
                    else None
                ),
                "eventClosed": event.get("closed"),
                "gammaResponseSha256": response_sha256,
            }
            audit_json = json.dumps(
                audit,
                ensure_ascii=False,
                separators=(",", ":"),
                sort_keys=True,
            )
            return {
                "value": value,
                "value_text": raw,
                "field": field,
                "candidates": candidates,
                "slug": slug,
                "request_url": str(response.url),
                "status_code": status,
                "fetched_at": fetched_at,
                "response_sha256": response_sha256,
                "event_closed": event.get("closed"),
                "audit_json": audit_json,
            }
        except GammaRequestError as exc:
            last_error = exc
            if not exc.retryable or attempt >= max_attempts:
                raise
            wait = retry_after_seconds(response) if response else None
            time.sleep(
                wait if wait is not None else SLEEP_SECONDS * attempt
            )
        except (
            requests.RequestException,
            ValueError,
            json.JSONDecodeError,
        ) as exc:
            last_error = exc
            if isinstance(exc, ValueError) or attempt >= max_attempts:
                raise GammaRequestError(
                    f"Gamma response rejected: {exc}",
                    status_code=(
                        int(response.status_code)
                        if response is not None
                        else None
                    ),
                    request_url=(
                        str(response.url)
                        if response is not None
                        else request_url
                    ),
                    response_text=(
                        response.text[:2000]
                        if response is not None
                        else None
                    ),
                    retryable=False,
                ) from exc
            time.sleep(SLEEP_SECONDS * attempt)

    raise GammaRequestError(
        f"Gamma request failed: {last_error}",
        request_url=request_url,
    )


In [ ]:
# ============================================================
# ROW CONSTRUCTION, VALIDATION, AND BACKFILL
# ============================================================

FETCH_CACHE: dict[tuple[str, int], dict[str, Any]] = {}


def cached_gamma_price_to_beat(
    underlying: str,
    bucket_start_ts: int,
) -> dict[str, Any]:
    key = (underlying.lower(), int(bucket_start_ts))
    if key not in FETCH_CACHE:
        FETCH_CACHE[key] = fetch_gamma_price_to_beat(
            underlying,
            bucket_start_ts,
        )
    return FETCH_CACHE[key]


def completed_flags(value: Any) -> tuple[Any, Any]:
    if isinstance(value, bool):
        return value, not value
    return pd.NA, pd.NA


def success_row(
    underlying: str,
    bucket_start_ts: int,
    opening: dict[str, Any],
    close: dict[str, Any] | None,
    close_error: str | None,
) -> dict[str, Any]:
    completed, incomplete = completed_flags(
        opening.get("event_closed")
    )
    fetched_at = opening["fetched_at"]
    audit = json.loads(opening["audit_json"])
    if close is not None:
        audit["nextBoundaryPriceToBeat"] = close["value_text"]
        audit["nextBoundarySlug"] = close["slug"]
        audit["nextBoundaryResponseSha256"] = close[
            "response_sha256"
        ]
    if close_error:
        audit["nextBoundaryError"] = close_error
    audit_json = json.dumps(
        audit,
        ensure_ascii=False,
        separators=(",", ":"),
        sort_keys=True,
    )
    return {
        "bucket_start_ts": int(bucket_start_ts),
        "underlying": underlying.lower(),
        "strike": float(opening["value"]),
        "slug": opening["slug"],
        "source": GAMMA_SOURCE,
        "fetched_at_utc": fetched_at.replace(
            microsecond=0
        ).isoformat().replace("+00:00", "Z"),
        "close_price": (
            float(close["value"])
            if close is not None
            else pd.NA
        ),
        "crypto_price_variant": GAMMA_VARIANT,
        "crypto_price_request_url": opening["request_url"],
        "crypto_price_http_status": opening["status_code"],
        "crypto_price_timestamp_ms": int(
            fetched_at.timestamp() * 1000
        ),
        "crypto_price_completed": completed,
        "crypto_price_incomplete": incomplete,
        "crypto_price_cached": pd.NA,
        "crypto_price_response_json": audit_json,
        "crypto_price_response_text": audit_json,
        "crypto_price_error": close_error or pd.NA,
    }


def failure_row(
    underlying: str,
    bucket_start_ts: int,
    exc: Exception,
) -> dict[str, Any]:
    fetched_at = utc_now()
    request_url = getattr(exc, "request_url", None) or (
        f"{GAMMA_EVENT_BY_SLUG_URL}/"
        f"{make_slug(underlying, bucket_start_ts)}"
    )
    return {
        "bucket_start_ts": int(bucket_start_ts),
        "underlying": underlying.lower(),
        "strike": pd.NA,
        "slug": make_slug(underlying, bucket_start_ts),
        "source": "gamma_field:error",
        "fetched_at_utc": fetched_at.replace(
            microsecond=0
        ).isoformat().replace("+00:00", "Z"),
        "close_price": pd.NA,
        "crypto_price_variant": GAMMA_VARIANT,
        "crypto_price_request_url": request_url,
        "crypto_price_http_status": getattr(
            exc, "status_code", None
        ),
        "crypto_price_timestamp_ms": int(
            fetched_at.timestamp() * 1000
        ),
        "crypto_price_completed": pd.NA,
        "crypto_price_incomplete": pd.NA,
        "crypto_price_cached": pd.NA,
        "crypto_price_response_json": pd.NA,
        "crypto_price_response_text": getattr(
            exc, "response_text", None
        ),
        "crypto_price_error": f"{type(exc).__name__}: {exc}",
    }


def fetch_row(
    underlying: str,
    bucket_start_ts: int,
) -> dict[str, Any]:
    try:
        opening = cached_gamma_price_to_beat(
            underlying,
            bucket_start_ts,
        )
        close = None
        close_error = None
        if FETCH_CLOSE_PRICE:
            try:
                close = cached_gamma_price_to_beat(
                    underlying,
                    bucket_start_ts + BUCKET_SECONDS,
                )
            except Exception as exc:
                close_error = (
                    f"next_boundary_{type(exc).__name__}: {exc}"
                )
        return success_row(
            underlying,
            bucket_start_ts,
            opening,
            close,
            close_error,
        )
    except Exception as exc:
        return failure_row(
            underlying,
            bucket_start_ts,
            exc,
        )


def format_duration(seconds: float | None) -> str:
    if seconds is None or not math.isfinite(seconds) or seconds < 0:
        return "--:--:--"
    total = int(round(seconds))
    hours, remainder = divmod(total, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


def progress_text(
    *,
    completed: int,
    total: int,
    successes: int,
    failures: int,
    started_at: float,
) -> str:
    elapsed = max(0.0, time.perf_counter() - started_at)
    remaining = max(0, total - completed)
    rate = completed / elapsed if completed > 0 and elapsed > 0 else 0.0
    eta = remaining / rate if rate > 0 else None
    percentage = (100.0 * completed / total) if total else 100.0
    return (
        f"Progress {completed:,}/{total:,} ({percentage:5.1f}%)"
        f" | left {remaining:,}"
        f" | ok {successes:,}"
        f" | failed {failures:,}"
        f" | elapsed {format_duration(elapsed)}"
        f" | ETA {format_duration(eta)}"
        f" | {rate:.2f} rows/s"
    )


def validate_known_bucket() -> None:
    if not RUN_KNOWN_BUCKET_VALIDATION:
        print("Known-bucket validation disabled.")
        return
    print("Running known official Price-to-Beat validation...")
    for (underlying, bucket_ts), expected in (
        KNOWN_BUCKET_VALIDATION.items()
    ):
        observed = cached_gamma_price_to_beat(
            underlying,
            bucket_ts,
        )["value"]
        difference = abs(observed - expected)
        print(
            f"  {underlying.upper()} {unix_to_iso_utc(bucket_ts)}: "
            f"Gamma={observed}, official display={expected}, "
            f"difference=${difference}"
        )
        if difference > KNOWN_BUCKET_TOLERANCE_USD:
            raise RuntimeError(
                "Known-bucket validation failed; no CSV was modified. "
                f"Observed {observed}, expected approximately {expected}."
            )


def backfill_asset(
    underlying: str,
) -> dict[str, Any]:
    output_path = DATA_PATH / STRIKE_FILE_NAMES[underlying]
    base_df = load_cache(output_path)
    buckets = bucket_range_from_dates(START_DATE, END_DATE)
    targets = build_repair_targets(
        base_df,
        underlying,
        buckets,
    )

    print()
    print(f"{underlying.upper()} target file: {output_path}")
    print(f"Configured buckets:    {len(buckets):,}")
    print(f"Repair targets:        {len(targets):,}")
    if not targets:
        return {
            "underlying": underlying,
            "final": base_df,
            "fetched": empty_cache(),
            "output_path": output_path,
            "backup_path": None,
        }

    fetched_rows: list[dict[str, Any]] = []
    failures_printed = 0
    success_count = 0
    failure_count = 0
    progress_started_at = time.perf_counter()
    last_progress_at = progress_started_at - PROGRESS_UPDATE_SECONDS
    for index, bucket_ts in enumerate(targets, start=1):
        row = fetch_row(underlying, bucket_ts)
        fetched_rows.append(row)
        if pd.notna(row["strike"]):
            success_count += 1
            if PRINT_EACH_SUCCESS:
                print(
                    f"[{index}/{len(targets)}] "
                    f"{unix_to_iso_utc(bucket_ts)} -> "
                    f"{row['strike']} ({row['source']})"
                )
        else:
            failure_count += 1
            if failures_printed < PRINT_FIRST_N_FAILURES:
                failures_printed += 1
                print()
                print(
                    f"[{index}/{len(targets)}] FAILED "
                    f"{unix_to_iso_utc(bucket_ts)}: "
                    f"{row['crypto_price_error']}"
                )

        if CHECKPOINT_EVERY and index % CHECKPOINT_EVERY == 0:
            checkpoint_df = merge_fetched_rows(
                base_df,
                pd.DataFrame(fetched_rows, columns=ALL_COLUMNS),
            )
            # Checkpoints use a separate file; the target remains untouched.
            checkpoint_path = output_path.with_suffix(
                output_path.suffix + ".gamma_twap60_checkpoint"
            )
            checkpoint_df.to_csv(checkpoint_path, index=False)

        if SLEEP_SECONDS > 0:
            time.sleep(SLEEP_SECONDS)

        now = time.perf_counter()
        progress_due = (
            index == 1
            or index == len(targets)
            or now - last_progress_at >= PROGRESS_UPDATE_SECONDS
        )
        if progress_due:
            message = progress_text(
                completed=index,
                total=len(targets),
                successes=success_count,
                failures=failure_count,
                started_at=progress_started_at,
            )
            final_update = index == len(targets)
            print(
                "\r" + message.ljust(150),
                end="\n" if final_update else "",
                flush=True,
            )
            last_progress_at = now

    fetched_df = normalize_cache(
        pd.DataFrame(fetched_rows, columns=ALL_COLUMNS)
    )
    final_df = merge_fetched_rows(base_df, fetched_df)
    backup_path = make_timestamped_backup(output_path)
    save_output(final_df, output_path)
    checkpoint_path = output_path.with_suffix(
        output_path.suffix + ".gamma_twap60_checkpoint"
    )
    if checkpoint_path.exists():
        checkpoint_path.unlink()

    return {
        "underlying": underlying,
        "final": final_df,
        "fetched": fetched_df,
        "output_path": output_path,
        "backup_path": backup_path,
    }


In [ ]:
# ============================================================
# OFFLINE SELF-TESTS — NO NETWORK AND NO FILE WRITES
# ============================================================

def run_offline_self_tests() -> None:
    slug = "btc-updown-5m-1789402200"
    event = {
        "slug": slug,
        "eventMetadata": {
            "priceToBeat": "78421.765923971878879232",
        },
        "markets": [{"conditionId": "condition"}],
    }
    normalized_event, market = normalize_event_payload(event, slug)
    value, field, raw, candidates = select_price_to_beat(
        normalized_event,
        market,
    )
    assert value == Decimal("78421.765923971878879232")
    assert raw == "78421.765923971878879232"
    assert field == "event.eventMetadata.priceToBeat"
    assert candidates[0]["field"] == field

    matching_event = {
        "priceToBeat": "100.00",
    }
    matching_market = {
        "metadata": {"price_to_beat": 100},
    }
    assert select_price_to_beat(
        matching_event, matching_market
    )[0] == Decimal("100.00")

    try:
        select_price_to_beat(
            {"priceToBeat": "100"},
            {"priceToBeat": "101"},
        )
    except ValueError as exc:
        assert "Conflicting" in str(exc)
    else:
        raise AssertionError("Conflicting PTB values were accepted")

    try:
        select_price_to_beat({}, {})
    except ValueError as exc:
        assert "no explicit priceToBeat" in str(exc)
    else:
        raise AssertionError("Missing PTB was accepted")

    assert source_needs_repair(
        "polymarket_crypto_price_api:fiveminute:openPrice"
    )
    assert source_needs_repair(
        "polymarket_crypto_price_api:http_400"
    )
    assert not source_needs_repair("gamma_field:priceToBeat")
    assert ALL_COLUMNS == [
        "bucket_start_ts",
        "underlying",
        "strike",
        "slug",
        "source",
        "fetched_at_utc",
        "close_price",
        "crypto_price_variant",
        "crypto_price_request_url",
        "crypto_price_http_status",
        "crypto_price_timestamp_ms",
        "crypto_price_completed",
        "crypto_price_incomplete",
        "crypto_price_cached",
        "crypto_price_response_json",
        "crypto_price_response_text",
        "crypto_price_error",
    ]
    print("Offline self-tests passed.")


if RUN_OFFLINE_SELF_TESTS:
    run_offline_self_tests()


In [ ]:
# ============================================================
# NETWORK VALIDATION, BACKFILL, AND FINAL SUMMARY
# ============================================================

# This validation runs before any CSV is modified.
validate_known_bucket()

results_by_underlying: dict[str, dict[str, Any]] = {}
for _underlying in SELECTED_UNDERLYINGS:
    results_by_underlying[_underlying] = backfill_asset(
        _underlying
    )

summary_rows: list[dict[str, Any]] = []
for _underlying, _result in results_by_underlying.items():
    _fetched = _result["fetched"]
    _success_count = int(_fetched["strike"].notna().sum())
    _failure_count = int(_fetched["strike"].isna().sum())
    summary_rows.append({
        "underlying": _underlying,
        "attempted_this_run": len(_fetched),
        "recovered_this_run": _success_count,
        "failed_this_run": _failure_count,
        "output_path": str(_result["output_path"]),
        "backup_path": (
            str(_result["backup_path"])
            if _result["backup_path"] is not None
            else None
        ),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

print()
print("Final schema check:")
for _underlying, _result in results_by_underlying.items():
    _path = _result["output_path"]
    _header = pd.read_csv(_path, nrows=0).columns.tolist()
    if _header != ALL_COLUMNS:
        raise RuntimeError(
            f"Schema changed unexpectedly for {_path}: {_header}"
        )
    print(f"  {_underlying.upper()}: exact 17-column schema confirmed")
